# __Dream interpretation RAG__

## __Installs & imports__

In [1]:
! pip install chromadb sentence-transformers

In [44]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import re
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
import string

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lesfabs/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## __Data preparation__

In [2]:
data = pd.read_csv('/Users/lesfabs/code/MartynaC/dreamscope/Notebooks/dream_notebook/dream_symbols_clean_v5.csv')
data.head()

,symbol_clean,slug,context,context_clean,meaning_clean
0,m,m,"To see the letter ""M"" in your dream",to see the letter m in your dream,suggests that there is something that you are ...
1,m&m's,mms,To see or eat M&M's in your dream,to see or eat mms in your dream,symbolizes lifes small but sweet rewards more ...
2,m&m's,mms,To dream that you are a giant talking M&M,to dream that you are a giant talking mm,suggests that you feel you are being mislead o...
3,macadamize,macadamize,"To see, walk or travel on a macadamized road i...",to see walk or travel on a macadamized road in...,suggests that you are standing on solid ground...
4,macaroni,macaroni,To dream that you are eating macaroni,to dream that you are eating macaroni,symbolizes comfort and ease the dream may be t...


In [3]:
data.context_clean.isnull().sum()

np.int64(7)

In [4]:
# Replace NaNs in  context_clean with the value of the slug column 
data.loc[data['context_clean'].isnull(), 'context_clean'] = data.loc[data['context_clean'].isnull(), 'slug']
data.context_clean.isnull().sum()

np.int64(0)

In [42]:
# Create chunks
data['chunk'] = data['context_clean'] + " " + data['meaning_clean']
data.sample(20)
data[data['slug']=='fire']

,symbol_clean,slug,context,context_clean,meaning_clean,chunk
5352,fire,fire,"Depending on the context of your dream, to see...",depending on the context of your dream to see ...,symbolize destruction passion desire illuminat...,depending on the context of your dream to see ...
5353,fire,fire,To dream that you are being burned by fire,to dream that you are being burned by fire,indicates that your temper is getting out of c...,to dream that you are being burned by fire ind...
5354,fire,fire,To dream that a house is on fire,to dream that a house is on fire,indicates that you need to undergo some transf...,to dream that a house is on fire indicates tha...
5355,fire,fire,To dream that you put out a fire,to dream that you put out a fire,signifies that you will overcome your obstacle...,to dream that you put out a fire signifies tha...
5356,fire,fire,To dream that you can start fire with your hands,to dream that you can start fire with your hands,represents anger that you are trying to repres...,to dream that you can start fire with your han...
5357,fire,fire,To dream that you can bend fire,to dream that you can bend fire,ers to your ability to control your anger,to dream that you can bend fire ers to your ab...
5358,fire,fire,Dreaming of an invisible fire,dreaming of an invisible fire,highlights a period of cleansing and purificat...,dreaming of an invisible fire highlights a per...
5359,fire,fire,To see a bird singing near a fire,to see a bird singing near a fire,ers to someone who is consumed by their passion,to see a bird singing near a fire ers to someo...
5360,fire,fire,Dreaming about fire and water together,dreaming about fire and water together,symbolizes a polarizing issue in your waking l...,dreaming about fire and water together symboli...


In [9]:
data[data['slug']=='laughing']

,symbol_clean,slug,context,context_clean,meaning_clean,chunk
2787,laughing,laughing,To hear laughing or dream that you are laughing,to hear laughing or dream that you are laughing,suggests that you need to lighten up and let g...,to hear laughing or dream that you are laughin...
2788,laughing,laughing,"To hear evil, demonic laughing in your dream",to hear evil demonic laughing in your dream,represents feelings of humiliation andor helpl...,to hear evil demonic laughing in your dream re...


In [7]:
data = data[data['context_clean'] != data['meaning_clean']].reset_index(drop=True)

## __MVP__

### __Embedding__

In [9]:
sentence_transformer_model = SentenceTransformer("all-mpnet-base-v2", device = 'mps')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
# One chunk
tryout_vector = sentence_transformer_model.encode(data['chunk'][0])
tryout_vector.shape

(768,)

In [11]:
# All chunks

embeddings = sentence_transformer_model.encode(data['chunk'], show_progress_bar=True)


Batches:   0%|          | 0/236 [00:00<?, ?it/s]

In [12]:
embeddings.shape

(7551, 768)

### __Setting up and querying collection__

In [13]:
# Initializing collection - slightly different from the method seen in previous challenges using LangChain 
# -> Here it's native ChromaDB, without an intermediary framework
client = chromadb.Client()

In [15]:
# To delete the collection if needed
#client.delete_collection("dream_symbols")

In [16]:
collection = client.create_collection("dream_symbols")

In [17]:
# Adding and indexing chunks

collection.add(
    documents=data['chunk'].tolist()[:5000],
    embeddings=embeddings[:5000],
    ids=[str(i) for i in range(5000)]
)

In [19]:
# We do it in two batches because of the limited batch size

collection.add(
    documents=data['chunk'].tolist()[5000:],
    embeddings=embeddings[5000:],
    ids=[str(i) for i in range(5000, 7551)]
)

In [20]:
collection.count()

7551

In [21]:
dreamer_text = "I was trapped in a cage with a slimy green strategy consultant. He was eating his computer."

In [22]:
searched_vector = sentence_transformer_model.encode(dreamer_text)
searched_vector.shape

(768,)

In [23]:
result=collection.query(
    searched_vector,
    n_results=6
)
result['documents']

[['to dream that you kill a leopard ers to success in your projects',
  'to dream that you are in a cage indicates that you are experiencing inhibitions and powerlessness in some areas of your life you are feeling restricted confined and restrained in a current relationship or business deal somebody may be keeping a short leash on you where you are lacking the freedom to act independently',
  'to dream that you are trapped or caught in a trap suggests that you are feeling confined and restricted in your job career health or a personal relationship you may be in a rut and are tired of the same daily monotony',
  'to dream that you catch a mouse in a mouse trap suggests that you are being taken advantage of',
  'to see or use a laptop in your dream represents your need to reach out and communicate with others in any circumstance',
  'to see a monitor lizard in your dream represents change and agility it also indicates your objectivity in a situation']]

### __Compiling query process in a dedicated function__ 

In [24]:
# Compile code in a dedicated function

def interpret(dream='',n_results=3):
    searched_vector = sentence_transformer_model.encode(dream)
    result=collection.query(
    searched_vector,
    n_results=n_results
    )
    return result['documents']

interpret(dreamer_text)

[['to dream that you kill a leopard ers to success in your projects',
  'to dream that you are in a cage indicates that you are experiencing inhibitions and powerlessness in some areas of your life you are feeling restricted confined and restrained in a current relationship or business deal somebody may be keeping a short leash on you where you are lacking the freedom to act independently',
  'to dream that you are trapped or caught in a trap suggests that you are feeling confined and restricted in your job career health or a personal relationship you may be in a rut and are tired of the same daily monotony']]

## __Trying to fine-tune the querying process__

### __Extracting keywords - Simple version__

In [25]:
# Initialize Model

tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')
model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3-mini-4k-instruct')

# Initialize pipeline

pipe = pipeline( 
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    )

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [26]:
# Define function to extract keywords

def extract_keywords(dreamer_text, n_keywords):
    # define prompt template
    prompt = f'You process a text input by extracting the {n_keywords} most significant keywords, ranked by significance. Expected format : a Python list of singular words, without any additional text'
    messages = [
        {'role': 'system', 'content': prompt},
        {'role': 'user', 'content': dreamer_text}
    ]

    # query model
    output = pipe(messages) 
    return output[0]['generated_text'][-1]['content']
    

In [31]:
result = extract_keywords(dreamer_text, 6)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [32]:
result

' ["cage", "slimy", "green", "strategy", "consultant", "computer"]'

In [33]:
keywords = re.findall(r"['\"]([^'\"]+)['\"]", result)


In [34]:
keywords

['cage', 'slimy', 'green', 'strategy', 'consultant', 'computer']

In [36]:
for word in keywords:
    searched_vector = sentence_transformer_model.encode(word)
    result = collection.query(
        searched_vector,
        n_results=1
    )
    print(word, result['documents'][0])

cage ['to watch a cage fight represents conflicting ideas or beliefs']
slimy ['to see or feel slime in your dream represents your inability to place your trust in someone the dream may be a metaphor for some one is a slimeball']
green ['the heart chakra is depicted by a green color and symbolizes pure and divine love for everyone and everything around you']
strategy ['to play foosball in your dream ers to a situation where you need to respond or act quickly you can not afford to lose site of your goal']
consultant ['to see a client in your dream ers to your position of power consider the position that you are in and the significance of the services that you are providing if you dream that you are a client then it means that you are looking for some sort of approval and validation of your ideas']
computer ['to see a computer in your dream symbolizes technology information and modern life new areas of opportunities are being opened to you alternatively computers represent a lack of indiv

### __Extracting keywords - Advanced version with preprocessing__

In [37]:
# Improve function to extract keywords -> Preprocess dream text

stop_words = set(stopwords.words('english'))

def preprocess(dream_text):
    # remove punctuation
    for punctuation in string.punctuation:
        dream_text = dream_text.replace(punctuation, ' ')

    # lowercase
    dream_text = dream_text.lower()
    
    # remove stopwords
    tokenized_text = word_tokenize(dream_text)
    clean_text = [word for word in tokenized_text if not word in stop_words]

    # lemmatize
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word) for word in clean_text]
    lemmatized_string = " ".join(lemmatized)
    
    return lemmatized_string


In [38]:
preprocessed_dream = preprocess(dreamer_text)
preprocessed_dream

'trapped cage slimy green strategy consultant eating computer'

In [39]:
result = extract_keywords(preprocessed_dream, 6)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [40]:
keywords = re.findall(r"['\"]([^'\"]+)['\"]", result)
keywords

['strategy', 'consultant', 'cage', 'eating', 'slimy', 'green']

In [41]:
for word in keywords:
    searched_vector = sentence_transformer_model.encode(word)
    result=collection.query(
        searched_vector,
        n_results=1
    )
    print(word, result['documents'][0])

strategy ['to play foosball in your dream ers to a situation where you need to respond or act quickly you can not afford to lose site of your goal']
consultant ['to see a client in your dream ers to your position of power consider the position that you are in and the significance of the services that you are providing if you dream that you are a client then it means that you are looking for some sort of approval and validation of your ideas']
cage ['to watch a cage fight represents conflicting ideas or beliefs']
eating ['to see someone eating with a fork denotes that all your current worries will be cleared up through the help of a friend']
slimy ['to see or feel slime in your dream represents your inability to place your trust in someone the dream may be a metaphor for some one is a slimeball']
green ['the heart chakra is depicted by a green color and symbolizes pure and divine love for everyone and everything around you']


__Problem with the approach :__ Working with keywords loses the context, so that when symbols have multiple context-dependant entries in the RAG it's pure chance whether we're going to get the right one. For instance, "Fire" as a keyword does not discriminate between "starting a fire", "being on fire", "fire truck"... 

In [ ]:
## TODO
# Try BerTopic https://maartengr.github.io/BERTopic/index.html // First out of the box, then trained on the symbols dictionary.
# Also try Named Entity Recognition (gleaner) to extract the most significant words from the text.
# Generate final response with LLM 

### __Querying the RAG - Alternate approach with embedding by sentences__

In [46]:
def tokenize_sentences(dreamer_text):
    sentences = sent_tokenize(dreamer_text)
    return sentences

In [48]:
sentences = tokenize_sentences(dreamer_text)

In [49]:
# Compile code in a dedicated function

def interpret_sentences(dream_sentences, n_results=3):
    interpretations = []
    for sentence in dream_sentences :
        searched_vector = sentence_transformer_model.encode(sentence)
        result=collection.query(
            searched_vector,
            n_results=n_results
        )
    interpretations.append(result['documents'][0])
    return interpretations

interpret(sentences)

[['to dream that you are in a cage indicates that you are experiencing inhibitions and powerlessness in some areas of your life you are feeling restricted confined and restrained in a current relationship or business deal somebody may be keeping a short leash on you where you are lacking the freedom to act independently',
  'to dream that you are trapped or caught in a trap suggests that you are feeling confined and restricted in your job career health or a personal relationship you may be in a rut and are tired of the same daily monotony',
  'to see or dream that you are in a greenhouse represents transformation you are experiencing some changes in your life brought about mainly as a result of your own doing it also suggests that you may be too overly controlling you want things done your way but in the process you may be isolating yourself'],
 ['to dream that your hard drive crashed indicates that you are being overwhelmed with information perhaps something is taking a emotional toll